In [0]:
from pyspark.sql.functions import first, rand, count

dates = spark.sql("SELECT explode(sequence(DATE'2024-01-01', DATE'2024-08-19', INTERVAL 1 DAY)) as calendar_date")
c_id = spark.sql("SELECT explode(sequence(1, 5000, 1)) as client_id")  
types = spark.sql("""SELECT concat("col_", colName) as col_name 
                    FROM (SELECT explode(sequence(1, 20, 1)) as colName)""")

dates = dates.repartition(100)
c_id = c_id.repartition(50)
types = types.repartition(1)


df_cartesian = c_id.crossJoin(dates.select("calendar_date")).crossJoin(types.select("col_name")).select("client_id", "calendar_date", "col_name")
df_cartesian = df_cartesian.withColumn("val", (rand() * 10).cast("int"))
df_grp = df_cartesian.groupBy("client_id", "calendar_date").pivot("col_name").agg(first("val").alias("val"))
display(df_grp.limit(1000))
df_grp.count() 

client_id,calendar_date,col_1,col_10,col_11,col_12,col_13,col_14,col_15,col_16,col_17,col_18,col_19,col_2,col_20,col_3,col_4,col_5,col_6,col_7,col_8,col_9
3614,2024-01-30,0,8,0,4,3,6,0,7,6,5,6,2,4,0,3,3,4,2,0,5
3614,2024-01-22,0,3,8,6,4,0,9,4,3,5,3,0,8,5,7,3,2,9,1,7
3614,2024-03-16,7,3,3,9,0,4,1,1,5,1,2,1,7,4,3,7,9,3,4,0
3614,2024-01-23,0,4,1,0,0,0,2,9,6,4,3,6,9,4,8,3,9,9,6,0
3614,2024-04-21,3,2,9,1,4,5,1,7,0,4,1,7,0,6,7,7,5,3,8,5
3614,2024-02-20,8,1,7,9,0,4,6,4,9,0,3,5,1,9,3,5,6,0,7,9
3614,2024-08-09,5,3,3,6,8,6,7,4,1,1,8,2,0,6,9,2,9,1,2,9
3614,2024-03-05,5,2,4,4,9,2,2,4,5,5,9,9,2,5,6,9,7,8,0,0
3614,2024-01-16,1,9,6,9,1,2,1,9,5,8,7,3,8,7,4,9,5,5,2,6
3614,2024-07-09,8,1,6,1,4,1,4,4,5,6,3,7,4,0,7,1,8,9,0,1


Out[3]: 1160000

In [0]:
from pyspark.sql.functions import first, rand
dates2 = spark.sql("SELECT explode(sequence(DATE'2024-01-01', DATE'2024-08-19', INTERVAL 1 DAY)) as calendar_date")
c_id2 = spark.sql("SELECT explode(sequence(1, 5000, 1)) as client_id") 
types2 = spark.sql("""SELECT concat("col_", colName) as col_name 
                    FROM (SELECT explode(sequence(1, 20, 1)) as colName)""")
dates2 = dates2.repartition(100)
c_id2 = c_id2.repartition(50)
types2 = types2.repartition(1)
df_cartesian2 = c_id2.crossJoin(dates2.select("calendar_date")).crossJoin(types2.select("col_name")).select("client_id", "calendar_date", "col_name")
df_cartesian2 = df_cartesian2.withColumn("val", (rand() * 10).cast("int"))
df_grp2 = df_cartesian2.groupBy("client_id", "calendar_date").pivot("col_name").agg(first("val").alias("val"))
display(df_grp2.limit(1000))
df_grp2.count() 

In [0]:
# INNER JOIN
inner_join_df = df_grp1.join(df_grp2, ["client_id", "calendar_date"], "inner").drop(df_grp2.calendar_date, df_grp2.client_id)

# LEFT JOIN
left_join_df = df_grp1.join(df_grp2, ["client_id", "calendar_date"], "left").drop(df_grp2.calendar_date, df_grp2.client_id)
